## Phones Scrapping

In [1]:
import selenium 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Phones List --
Phones = []

## -- Scrapping --
driver = webdriver.Chrome(options=options)
driver.get("https://www.amazon.eg/-/en/b/?_encoding=UTF8&node=21832883031&bbn=18018102031&ref_=Oct_d_odnav_d_21832868031_2&pd_rd_w=MfMw4&content-id=amzn1.sym.7f7c0e7a-4694-4a1c-b568-1063e9e5cd31&pf_rd_p=7f7c0e7a-4694-4a1c-b568-1063e9e5cd31&pf_rd_r=NBJ9KT5HK4KMCSTYG044&pd_rd_wg=74z8s&pd_rd_r=9fa37c3e-1894-4301-8ca8-6aeaf3784ea2")
time.sleep(5)
driver.find_element(By.ID, 'sp-cc-accept').click()
time.sleep(5)
driver.find_element(By.ID, 'apb-desktop-browse-search-see-all').click()
time.sleep(5)
page = 2

while True:

    cards = driver.find_elements(By.CSS_SELECTOR,'div[data-component-type="s-search-result"]')
    page_products = 0

    for card in cards:

        try:
            Name = card.find_element(By.CSS_SELECTOR, 'h2').get_attribute('aria-label')
            Link = card.find_element(By.CSS_SELECTOR, 'a.a-link-normal[href*="/dp/"]').get_attribute('href')
            Image = card.find_element(By.CSS_SELECTOR, 'img.s-image').get_attribute('src')

        except NoSuchElementException:
            continue

        try:
            Rating = card.find_element(By.CSS_SELECTOR,'[data-cy="reviews-block"] span.a-size-small.a-color-base').text

        except NoSuchElementException:
            Rating = "Not Available"

        try:
            Reviews = card.find_element(By.CSS_SELECTOR,'[data-cy="reviews-block"] a[aria-label$="ratings"]').get_attribute('aria-label')

        except NoSuchElementException:
            Reviews = "Not Available"

        try:
            Price = card.find_element(By.CLASS_NAME, 'a-price-whole').text

        except NoSuchElementException:
            Price = "Product Not Available"

        Phones.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Rating': Rating,
            'Reviews': Reviews,
            'Price': Price
        })

        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Phones)}"
    )

    try:
        next_page = driver.find_element(By.CSS_SELECTOR,'a.s-pagination-next')

        if 's-pagination-disabled' in next_page.get_attribute('class'):
            print("Scraping finished. No more pages available.")
            break

        next_page.click()
        page += 1
        time.sleep(5)

    except NoSuchElementException:
        print("Next page button not found. Check the pagination selector.")
        break


## To CSV & Parquet Files 

In [4]:
import pandas as pd 

In [5]:
df_Phones = pd.DataFrame(Phones)

In [6]:
df_Phones.head()

,Name,Link,Image,Rating,Reviews,Price
0,"HONOR X9d 5G Smartphone, 8GB RAM + 256GB Stora...",https://www.amazon.eg/-/en/HONOR-Smartphone-Su...,https://m.media-amazon.com/images/I/81tKzzryLr...,4.7,34 ratings,"19,999"
1,"Xiaomi Redmi A7 Pro Smartphone 4GB+128GB 4G, B...",https://www.amazon.eg/-/en/Xiaomi-Smartphone-O...,https://m.media-amazon.com/images/I/61MrVEB8Dw...,3.0,20 ratings,"9,499"
2,"XIAOMI REDMI 17 Smartphone, 6GB RAM + 256GB, 6...",https://www.amazon.eg/-/en/Smartphone-G91-Ultr...,https://m.media-amazon.com/images/I/612x88aPFk...,3.8,12 ratings,"13,999"
3,Apple iPhone 17 Pro Max (256 GB) - Cosmic Oran...,https://www.amazon.eg/-/en/Apple-iPhone-Pro-Ma...,https://m.media-amazon.com/images/I/61yXZ6sG6G...,3.8,77 ratings,"93,929"
4,"realme C100i AI Phone 4G Smartphone, 128GB Sto...",https://www.amazon.eg/-/en/realme-C100i-Smartp...,https://m.media-amazon.com/images/I/61BIeEZ9fj...,Not Available,Not Available,Product Not Available


In [7]:
df_Phones.to_csv("Phones.csv" , index = False )

In [8]:
df_Phones.to_parquet("Phones.parquet" , index = False )